In [1]:
import time
start_time=time.time()

In [2]:
import numpy as np
import pandas as pd
import sys
from pathlib import Path
import astropy.units as u
from astropy.io import fits

sys.path.append(str(Path().resolve() / 'py_modules')) # 1 level up = project root
from turb_utils import make_extended, make_3dfield

In [3]:
# Physical parameters

# Fake map parameters
N          = 512
ra         = N / 16 # Artificial correlation length
m2D        = 0.85   # for recovering m as 1.00 in the projected field (k = 2 + m2D) non emissivity fluctuation case
ellip      = 0.5,
theta      = 45,
randomseed_vv = 1894_04_25,

# Fake obs parameters
s0    = 0
noise = 0
pc    = 1
pix   = 1

In [10]:
name  = 'fake_map_mod_finite_m' +str(int((2+m2D)*100))+'_r' +str(int(ra))+ '_N' +str(N)
name

'fake_map_mod_finite_m285_r32_N512'

In [4]:
# Tapered map
vmap = make_extended(
    N,
    powerlaw           = 2.0 + m2D,
    ellip              = ellip,
    theta              = theta,
    correlation_length = ra,
    randomseed        = randomseed_vv,
)

sig = vmap.std()
vmap /= sig

sig

1.8482080520449609

In [5]:
hdu = fits.PrimaryHDU(data=vmap)
hdul = fits.HDUList([hdu])
hdr = hdu.header

In [11]:
# --- Add properties to the header ---
hdr = hdu.header
hdr["AUTHOR"]   = ("J. Garcia-Vazquez", "File creator")
hdr["m2D"]      = (m2D, "k = 2 + m2D")
hdr["ra"]       = (ra, "Artificial correlation length")
hdr["sig"]      = (sig, "Standard deviation artificial map")
hdr["sig2"]     = (sig**2, "Variance artificial map")
hdr["s0"]       = (s0, "Atmospheric seeing")
hdr["noise"]    = (noise, "Instrumental noise")
hdr["box_size"] = (N, "Observational box_size")
hdr["pc"]       = (pc, "parsec convertion")
hdr["pix"]      = (pix, "Instrument pixel scale")
hdr["LINE"]     = ("H_I-6563", "Emission line name")
hdr["BINSIZE"]  = (0, "Spatial binning factor")
hdr["SEED"]     = (randomseed_vv, "Random seed")

# Add a history / comment line
#hdr.add_history("Processed with custom Python pipeline v0.1")
#hdr.add_comment("Velocity map cleaned and masked")

In [12]:
hdul.info()

Filename: (No file associated with this HDUList)
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU      18   (512, 512)   float64   


In [13]:
print(repr(hdr))

SIMPLE  =                    T / conforms to FITS standard                      
BITPIX  =                  -64 / array data type                                
NAXIS   =                    2 / number of array dimensions                     
NAXIS1  =                  512                                                  
NAXIS2  =                  512                                                  
EXTEND  =                    T                                                  
AUTHOR  = 'J. Garcia-Vazquez'  / File creator                                   
M2D     =                 0.85 / k = 2D + m2D                                   
RA      =                 32.0 / Artificial correlation length                  
SIG     =   1.8482080520449609 / Standard deviation artificial map              
SIG2    =   3.4158730036438287 / Variance artificial map                        
S0      =                    0 / Atmospheric seeing                             
NOISE   =                   

In [14]:
hdul.writeto(name + '.fits', overwrite=True)

In [10]:
print("--- %s seconds ---" % (time.time()-start_time))

--- 2.1949355602264404 seconds ---
